In [1]:
import os
os.environ['HF_HOME'] = '/projectnb/vkolagrp/skowshik/.cache/'
import sys
import logging
import argparse
import warnings
warnings.filterwarnings("ignore")
import yaml
import torch
import json
import inspect
import pandas as pd
import torch.distributed as dist
import torchtune

from tqdm import tqdm
from torch.nn.parallel import DistributedDataParallel as DDP
from datetime import datetime, timedelta
from huggingface_hub import login
from lib.model_loader import load_model
from lib.trainer_loader import load_trainer
from lib.data_loader import data_loader_
from utils.utils import CustomStream, load_config
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, AutoConfig, AutoModel
from lib.model_loader_eval import load_model_eval, load_model_eval_vllm, save_merged_model, load_model_with_llama_proj
from utils.utils import load_json, load_csv
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from safetensors.torch import load_file
from tokenizers import AddedToken
from peft import LoraConfig, TaskType, PeftModel, get_peft_model
from lib.modeling_llama import LlamaVisionForCausalLM, LlamaForCausalLM

In [3]:
# config = load_config(file_name="../../code/training/config/config_imaging.yml")
config = load_config(file_name="../../code/training/config/config.yml")
sysmsg = config.get("sysmsg")
dataset_path = "../../data/0825/nacc_unique_with_llama_summaries_test.json"
# dataset_path = "../../data/Imaging/embedding_data/imaging_model_dataset_test.csv"
data = load_json(dataset_path)
# data = load_csv(dataset_path)
n_devices = 1
# lora_path = '../../ckpt/fine_tuned_v1/checkpoint-52000/'
# lora_path = '../../ckpt/fine_tuned_v2_img/checkpoint-3870/'
max_new_tokens = config.get("max_new_tokens")
hf_write_token = config.get("hf_write_token")

In [ ]:
data[0]

In [7]:
for entry in data[::-1]:
    if 'amyloid' in entry['patient_LLAMA_SUMMARY']:
        print(entry['patient_LLAMA_SUMMARY'])
        break

**Medical History Summary**

**Subject Demographics**

* Age: 73 years
* Sex: Female
* Race: White
* Marital status: Married
* Education: 14 years
* Living situation: Lives with spouse or partner

**Family History**

* No report of father with cognitive impairment
* No report of mother with cognitive impairment
* No family history of Alzheimer's disease (AD) or frontotemporal lobar degeneration (FTLD)
* No family history of other neurodegenerative disorders

**Subject Medications**

* Total number of medications: 7
* Current use of FDA-approved medication for Alzheimer's disease symptoms: Yes
* Medications used within two weeks of visit: ALBUTEROL, MESALAMINE, CHOLECALCIFEROL, CETIRIZINE, DONEPEZIL, RALOXIFENE, FLUTICASONE NASAL

**Physical Examination**

* Weight: 120 lbs
* Height: 64 inches
* Body mass index (BMI): 20.6
* Blood pressure: 169/93 mmHg
* Resting heart rate: 73 beats per minute

**Neuropsychiatric Inventory Questionnaire (NPI-Q)**

* Anxiety severity: Severe
* Agitation 

In [3]:
from huggingface_hub import login
login(token = hf_write_token)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /projectnb/vkolagrp/skowshik/.cache/token
Login successful


In [4]:
model = LlamaVisionForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3-8B-Instruct",
    use_auth_token=hf_write_token,
    device_map="auto",
    torch_dtype="bfloat16",
    cache_dir=config.get("cache_dir"),
    k=1
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.31it/s]
Some weights of LlamaVisionForCausalLM were not initialized from the model checkpoint at meta-llama/Meta-Llama-3-8B-Instruct and are newly initialized: ['model.img_model.swinunetr.decoder1.conv_block.conv1.conv.weight', 'model.img_model.swinunetr.decoder1.conv_block.conv2.conv.weight', 'model.img_model.swinunetr.decoder1.conv_block.conv3.conv.weight', 'model.img_model.swinunetr.decoder1.transp_conv.conv.weight', 'model.img_model.swinunetr.decoder2.conv_block.conv1.conv.weight', 'model.img_model.swinunetr.decoder2.conv_block.conv2.conv.weight', 'model.img_model.swinunetr.decoder2.conv_block.conv3.conv.weight', 'model.img_model.swinunetr.decoder2.transp_conv.conv.weight', 'model.img_model.swinunetr.decoder3.conv_block.conv1.conv.weight', 'model.img_model.swinunetr.decoder3.conv_block.conv2.conv.weight', 'model.img_model.swinunetr.decoder3.conv_block.conv3.conv.weight', 'model.img_model.swinunetr.decoder3.transp_conv

In [5]:
tokenizer = AutoTokenizer.from_pretrained(
        "meta-llama/Meta-Llama-3-8B-Instruct",
        use_auth_token=hf_write_token,
        device_map="auto",
        cache_dir=config.get("cache_dir"),
    )

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
model.set_tokenizer(tokenizer)

In [7]:
messages = [
    [
        {"role": "user", "content": f"Hello!"}
    ],
]

### Load model

#### VLLM

In [ ]:
# llm, tokenizer = load_model_eval_vllm(config, n_devices=2, enable_lora=False, torch_dtype="bfloat16", new_model_path="skowshik/llama3-8b-coco-vision")
llm, tokenizer = load_model_eval_vllm(config, n_devices=2, enable_lora=False, torch_dtype="bfloat16", new_model_path=config.get("hub_final_model_id"))

#### Model

In [ ]:
# model, tokenizer = load_model(config, torch_dtype=torch.bfloat16, vision=True)
# model, tokenizer = load_model_eval(config, base_model="skowshik/llama3-8b-coco-vision", lora_path=None, torch_dtype=torch.bfloat16, vision=True)
# model, tokenizer = load_model_eval(config, base_model=config.get("hub_final_model_id"), torch_dtype=torch.bfloat16, vision=True)
model, tokenizer = load_model_eval(config, base_model=config.get("model_name"), lora_path=config.get("hub_model_id"), torch_dtype=torch.bfloat16, vision=True)
# model.get_input_embeddings()

In [5]:
model.model.set_tokenizer(tokenizer)

### Set prompt

#### ADRD

In [6]:
nacc = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/new_nacc_unique_type_3.csv")

#### from json

In [11]:
# index = 0 # DE with LBD
# index = 2 # NC
index = 6 # MCI with VD
# index = 16 # DE with AD
# index = 14 # FTD
data[index]['NACCUDSD']

3

In [14]:
data[index]['instruction']

'Provide a cognitive diagnosis for a patient presenting with the following information.'

In [45]:
messages = [
    [
        # {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"### Instruction:\n{data[index]['instruction']}\n\n### Input:\n{data[index]['patient_LLAMA_SUMMARY']}"}
    ],
    
    [
        {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"### Instruction:\nProvide a differential diagnosis of this patient. \n\n### Input:\n{data[index]['patient_LLAMA_SUMMARY']}"}
    ],
]

#### from csv

In [6]:
import re
import numpy as np

def add_image_tokens(df):
    pattern = r'(/\S+\.npy)'
    all_matches = []

    # Iterate over each row in the DataFrame
    for i, row in df.iterrows():
        # Find all matches of the pattern in the 'user' column
        matches = re.findall(pattern, row['user'])
        all_matches += matches
    
    all_matches = list(set(all_matches))
    for match in all_matches:
        try:
            x = np.load(match, mmap_mode='r')
        except:
            print(match)
            raise ValueError
    print(len(all_matches))
    return all_matches

all_matches = add_image_tokens(data)
# all_matches.append('/projectnb/vkolagrp/datasets/NACC/MRI/skullstripped/NACC004099/fname-NACC004099_128401136192134176253428319421300140307886_mo-4_dy-27_yr-2011/T1/3D/1.2.840.113619.2.134.1762534283.1942.1300140307.908_stripped.nii')

new_tokens = set()
if all_matches:
    for matched_string in all_matches:
        token = AddedToken(matched_string, lstrip=False, rstrip=False, normalized=False, special=False)
        new_tokens.add(token)
    tokenizer.add_tokens(list(new_tokens))
    model.resize_token_embeddings(len(tokenizer))

282


In [7]:
def process_user_message(user_msg, reserved_tokens, tokenizer):
    # Define the regular expression pattern for words ending with .nii
    pattern = r'(/\S+\.npy)'
    
    # Define the replacement function
    # It adds "start_of image" before the word, quotes around the word, and "{reserved_tokens}end_of_mri" after
    def replace_func(match):
        matched_string = match.group(1)
        
        # Print the matched string
        # print("Matched string:", matched_string)
        return f'<|start_of_mri|>{matched_string}{reserved_tokens}<|end_of_mri|>'
    
    # Use re.sub to replace all occurrences in the input message
    processed_msg = re.sub(pattern, replace_func, user_msg)
    # print(processed_msg)
    
    return processed_msg

In [17]:
index = 2
# index = 2
row = data.iloc[index]
print(row['NACCID'])
print(row['assistant'])
reserved_tokens = '<|reserved_mri_token|>' * (config.get('k') - 1)
user_msg = process_user_message(row['user'], reserved_tokens, tokenizer)

NACC024275
Yes, the MRI shows extensive white-matter hyperintensity.


In [29]:
mod_user_msg = user_msg.replace("T1 MRI ", "").replace("T2 MRI ", "").replace("SWI MRI ", "").replace("FLAIR MRI ", "")
messages = [
    [
        {"role": "system", "content": sysmsg},
        {"role":"user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\n{row['instruction']} The MRI images are provided between tokens <|start_of_mri|> and <|end_of_mri|>.\n\n### Input:\n{user_msg}"},
    ],
    
    [
        # {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat do you see in the image? The images are provided between tokens <|start_of_mri|> and <|end_of_mri|>.\n\n### Input:\n{mod_user_msg}"},
        
    ],
    
    
]

In [30]:
messages

[[{'role': 'system', 'content': 'You are a helpful radialogist AI assistant.'},
  {'role': 'user',
   'content': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nDoes this MRI imaging study show extensive white-matter hyperintensity? The MRI images are provided between tokens <|start_of_mri|> and <|end_of_mri|>.\n\n### Input:\nT1 MRI image: <|start_of_mri|>/projectnb/vkolagrp/datasets/NACC/MRI/swin_emb/NACC024275/fname-8683_NACC024275_20170517_mo-5_dy-17_yr-2017/T1/3D/1.3.46.670589.11.24071.5.0.5896.2017051711163134121_stripped.npy<|reserved_mri_token|><|reserved_mri_token|><|reserved_mri_token|><|reserved_mri_token|><|end_of_mri|>'}],
 [{'role': 'user',
   'content': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat do you 

#### COCO

In [4]:
coco_3d = pd.read_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/image_captioning_dataset/All_Caption_ID_3D.csv')

In [5]:
coco_3d.iloc[412534]

image_id       /projectnb/vkolagrp/skowshik/foundation_adrd/i...
name                                             ski person tree
description    a snowboarder is in the air as he attempts a s...
Name: 412534, dtype: object

In [6]:
index = 0

In [7]:
img_emb = coco_3d.iloc[index]['image_id'].replace('train2014/', 'train2014_emb/').replace('.jpg', '.npy')
messages = [
    [
        # {"role": "system", "content": sysmsg},
        {"role": "user", "content": f"### Instruction:\nDescribe this image. \n\n### Input:\n<|start_of_mri|>{img_emb}<|end_of_mri|>"}
    ],
    
]

### VLLM testing

ADD "unable to complete" for 996, 997 etc codes

In [ ]:
print(data[index]['patient_LLAMA_SUMMARY'])

In [46]:
prompts = []

for message in messages:
    input_ids = tokenizer.apply_chat_template(
        message,
        add_generation_prompt=True,
        tokenize=False,
        # return_tensors="pt"
    )
    
    prompts.append(input_ids)

In [47]:
# prompts = [torchtune.data.AlpacaInstructTemplate.format(sample={"instruction": "Provide a cognitive diagnosis in ONE WORD for a patient presenting with the following information.", "input": data[index]['patient_LLAMA_SUMMARY']})]

In [52]:
responses = []
sampling_params = SamplingParams(
    temperature=0.2,
    max_tokens=max_new_tokens,
    # stop=stop_tokens
)

completions = llm.generate(
    prompts=prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest("adapter", 1, lora_path)
)

for i, output in enumerate(completions):
    temp_gen = output.outputs[0].text
    responses.append(temp_gen)

Processed prompts: 100%|██████████| 2/2 [00:05<00:00,  2.99s/it, Generation Speed: 157.43 toks/s]


In [53]:
print(responses[1])

Based on the provided patient information, I will provide a differential diagnosis for the patient's cognitive impairment.

**Primary Diagnosis:** Alzheimer's disease (AD)

**Secondary Diagnoses:**

1. Vascular dementia (possible)
2. Lewy body disease (possible)
3. Frontotemporal dementia (possible)
4. Other etiologies (e.g., traumatic brain injury, CNS neoplasm, prion disease)

**Reasoning:**

1. The patient's age (93 years) and APOE genotype (e3 e3) suggest a higher risk for AD.
2. The patient's cognitive impairment is characterized by a gradual decline in memory, attention, and executive function, which is consistent with AD.
3. The patient's MMSE score of 23 is consistent with mild to moderate cognitive impairment.
4. The patient's Boston Naming Test score of 19 is also consistent with mild to moderate cognitive impairment.
5. The patient's GDS score of 3 suggests that the patient is not depressed, which is important to consider in the differential diagnosis of cognitive impairment

### Model testing

In [8]:
prompt = tokenizer.apply_chat_template(
    messages[0], 
    # tokenize=False, 
    return_tensors="pt",
    add_generation_prompt=True
).to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
        prompt,
        max_new_tokens=100,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=terminators,
        do_sample=True,
        temperature=0.2,
    )
response = outputs[0][prompt.shape[-1]:]
print(tokenizer.decode(response, skip_special_tokens=True))
# # # # tokenizer.decode(response, skip_special_tokens=True)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token.As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [22]:
row['assistant']

'Yes, the MRI shows extensive white-matter hyperintensity.'

In [30]:
# from huggingface_hub import HfApi

# api = HfApi()
# api.upload_file(
#     path_or_fileobj="/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/ckpt/fine_tuned_v4/checkpoint-27000/ggml-lora-f16.gguf",
#     path_in_repo="llama3-8b-adrd.gguf",
#     repo_id="skowshik/llama3-8b-adrd",
#     repo_type="model",
# )


In [4]:
# file_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/ckpt/coco_test_1/model-00001-of-00004.safetensors"
# loaded = load_file(file_path, device="cuda:1")

### Other testing

In [2]:
file1 = load_file("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/ckpt/fine_tuned_v2_img/model-00004-of-00004.safetensors")

In [4]:
file1['llama_proj.0.weight']

tensor([[ 0.0227, -0.0023, -0.0398,  ..., -0.0271,  0.0332,  0.0019],
        [-0.0016, -0.0160,  0.0049,  ...,  0.0074, -0.0127, -0.0089],
        [ 0.0093, -0.0183,  0.0010,  ...,  0.0134, -0.0300, -0.0084],
        ...,
        [-0.0087, -0.0106, -0.0008,  ..., -0.0103,  0.0256,  0.0105],
        [-0.0240,  0.0157, -0.0211,  ..., -0.0078, -0.0111, -0.0025],
        [-0.0210, -0.0325,  0.0308,  ..., -0.0223,  0.0067, -0.0261]],
       dtype=torch.bfloat16)

In [6]:
file1 = load_file("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/ckpt/fine_tuned_v4_img_with_lora_test/model-00001-of-00004.safetensors")
file2 = load_file("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/ckpt/fine_tuned_v3_img_with_lora/model-00001-of-00004.safetensors")
orig = load_file("/projectnb/vkolagrp/skowshik/.cache/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/e1945c40cd546c78e41f1151f4db032b271faeaa/model-00001-of-00004.safetensors")

In [29]:
tokenizer.encode("Grave")

[128000, 6600, 525]

In [36]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

x_normalized = normalize([model.get_input_embeddings().weight[525].to(torch.float32).detach().cpu().numpy()])[0]
y_normalized = normalize([model.get_input_embeddings().weight[128256].to(torch.float32).detach().cpu().numpy()])[0]

# Assume embedding1 and embedding2 are your vectors for T1 and SWI MRI images respectively
similarity = cosine_similarity([x_normalized], [y_normalized])
similarity

array([[-0.02440419]])

In [30]:
model.get_input_embeddings().weight[6600].to('cpu')

tensor([-0.0024,  0.0092,  0.0052,  ...,  0.0124,  0.0020, -0.0136],
       dtype=torch.bfloat16, grad_fn=<ToCopyBackward0>)

In [31]:
model.get_input_embeddings().weight[128256].to('cpu')

tensor([-0.0312,  0.0118,  0.0021,  ..., -0.0159, -0.0040, -0.0255],
       dtype=torch.bfloat16, grad_fn=<ToCopyBackward0>)

In [89]:
orig['model.embed_tokens.weight'][9642]

tensor([ 0.0104, -0.0026, -0.0057,  ...,  0.0209,  0.0103, -0.0087],
       dtype=torch.bfloat16)

In [90]:
file1['embed_tokens.weight'][9642]

tensor([ 1.2756e-02,  6.5002e-03, -1.0437e-02,  ...,  2.3071e-02,
         1.1981e-05, -7.7209e-03], dtype=torch.bfloat16)

In [91]:
file2['embed_tokens.weight'][9642]

tensor([ 0.0104, -0.0026, -0.0057,  ...,  0.0209,  0.0103, -0.0087],
       dtype=torch.bfloat16)

In [9]:
torch.equal(orig['model.embed_tokens.weight'][9642], file1['embed_tokens.weight'][9642])

False

In [10]:
torch.equal(model.get_input_embeddings().weight[9642].to('cpu'), file1['embed_tokens.weight'][9642])

True